# Charles Bridge Image Analysis Testing

Testing the new structured prompt for Charles Bridge CCTV image analysis using GPT-4 Vision API.

In [9]:
import os
import base64
import requests
import json
from pathlib import Path
from PIL import Image
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Azure OpenAI Configuration
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_MODEL = os.getenv("AZURE_OPENAI_MODEL")

# Build Azure API URL
if AZURE_OPENAI_ENDPOINT:
    API_URL = f"{AZURE_OPENAI_ENDPOINT}openai/deployments/{AZURE_OPENAI_MODEL}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
else:
    API_URL = None

# Check configuration
if not all([AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_MODEL]):
    print("⚠️ Missing Azure OpenAI configuration in .env file")
    print("Required: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_MODEL")
else:
    print("✅ Azure OpenAI configuration loaded")
    print(f"🔗 Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"📋 Model: {AZURE_OPENAI_MODEL}")
    print(f"📅 API Version: {AZURE_OPENAI_API_VERSION}")

✅ Azure OpenAI configuration loaded
🔗 Endpoint: https://openaiendpoint-uk-1.openai.azure.com/
📋 Model: gpt-4.1
📅 API Version: 2025-04-01-preview


In [10]:
# Load the structured prompt from the file
def load_charles_bridge_prompt():
    """Load the structured prompt from the charles_bridge.md file"""
    prompt_file = Path("../prompts/charles_bridge.md")
    
    if not prompt_file.exists():
        print(f"❌ Prompt file not found: {prompt_file}")
        return None
    
    with open(prompt_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    print("✅ Charles Bridge prompt loaded successfully")
    return content

# Load the prompt
charles_bridge_prompt_content = load_charles_bridge_prompt()

print("📋 Loaded structured prompt from file")
print("🔄 Using markdown prompt directly for API calls")

✅ Charles Bridge prompt loaded successfully
📋 Loaded structured prompt from file
🔄 Using markdown prompt directly for API calls


In [11]:
def encode_image_to_base64(image_path):
    """Convert image to base64 string for API"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_bridge_image(image_path, prompt):
    """
    Analyze Charles Bridge camera image using Azure OpenAI GPT-4 Vision API
    
    Args:
        image_path (str): Path to the image file
        prompt (str): Custom prompt for analysis (uses charles_bridge_prompt if None)
    
    Returns:
        dict: API response with analysis
    """
    if not AZURE_OPENAI_API_KEY or not API_URL:
        return {"error": "Azure OpenAI configuration missing"}
    
    # Encode image
    try:
        base64_image = encode_image_to_base64(image_path)
    except Exception as e:
        return {"error": f"Failed to encode image: {str(e)}"}
    
    # Prepare Azure API request headers
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY
    }
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        "max_tokens": 1000,
        "temperature": 0.1  # Low temperature for consistent analysis
    }
    
    try:
        print(f"🔄 Analyzing image: {Path(image_path).name}")
        response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        analysis_text = result["choices"][0]["message"]["content"]
        
        # Try to parse JSON from the response
        try:
            # Find JSON in the response
            json_start = analysis_text.find('{')
            json_end = analysis_text.rfind('}') + 1
            if json_start >= 0 and json_end > json_start:
                json_str = analysis_text[json_start:json_end]
                parsed_analysis = json.loads(json_str)
            else:
                parsed_analysis = {"raw_response": analysis_text}
        except json.JSONDecodeError:
            parsed_analysis = {"raw_response": analysis_text}
        
        return {
            "success": True,
            "analysis": parsed_analysis,
            "raw_response": analysis_text,
            "usage": result.get("usage", {}),
            "timestamp": datetime.now().isoformat(),
            "image_path": str(image_path),
            "model": AZURE_OPENAI_MODEL
        }
        
    except requests.exceptions.RequestException as e:
        return {"error": f"Azure API request failed: {str(e)}"}
    except Exception as e:
        return {"error": f"Unexpected error: {str(e)}"}

print("✅ Analysis function ready")

✅ Analysis function ready


In [12]:
# Get list of test images
image_dir = Path("dataset/charles_bridge/images")

dataset = json.load(open("dataset/charles_bridge/annotations/annotations.json"))
annotations = dataset.get("annotations", [])

annotations

[{'image_names': ['camera_101200_20251129_120603.jpg',
   'camera_101201_20251129_120604.jpg'],
  'crowdedness_score': 9},
 {'image_names': ['camera_101200_20251125_095232.jpg',
   'camera_101201_20251125_095233.jpg'],
  'crowdedness_score': 2}]

In [13]:
def analyze_bridge_images_pair(image_paths, prompt):
    """
    Analyze multiple Charles Bridge camera images using Azure OpenAI GPT-4 Vision API
    Perfect for analyzing both sides of the bridge simultaneously
    
    Args:
        image_paths (list): List of paths to image files (e.g., both sides of bridge)
        prompt (str): Prompt for analysis
    
    Returns:
        dict: API response with analysis
    """
    if not AZURE_OPENAI_API_KEY or not API_URL:
        return {"error": "Azure OpenAI configuration missing"}
    
    # Encode all images
    try:
        encoded_images = []
        for image_path in image_paths:
            base64_image = encode_image_to_base64(image_path)
            encoded_images.append(base64_image)
    except Exception as e:
        return {"error": f"Failed to encode images: {str(e)}"}
    
    # Prepare Azure API request headers
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY
    }
    
    # Build content array with text prompt and all images
    content = [{"type": "text", "text": prompt}]
    
    for i, base64_image in enumerate(encoded_images):
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}",
                "detail": "high"
            }
        })
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": content
            }
        ],
        "max_tokens": 1500,  # Increased for multiple images
        "temperature": 0.1
    }
    
    try:
        image_names = [Path(img).name for img in image_paths]
        print(f"🔄 Analyzing {len(image_paths)} images: {', '.join(image_names)}")
        
        response = requests.post(API_URL, headers=headers, json=payload, timeout=90)
        response.raise_for_status()
        
        result = response.json()
        analysis_text = result["choices"][0]["message"]["content"]
        
        # Try to parse JSON from the response
        try:
            json_start = analysis_text.find('{')
            json_end = analysis_text.rfind('}') + 1
            if json_start >= 0 and json_end > json_start:
                json_str = analysis_text[json_start:json_end]
                parsed_analysis = json.loads(json_str)
            else:
                parsed_analysis = {"raw_response": analysis_text}
        except json.JSONDecodeError:
            parsed_analysis = {"raw_response": analysis_text}
        
        return {
            "success": True,
            "analysis": parsed_analysis,
            "raw_response": analysis_text,
            "usage": result.get("usage", {}),
            "timestamp": datetime.now().isoformat(),
            "image_paths": image_paths,
            "image_count": len(image_paths),
            "model": AZURE_OPENAI_MODEL
        }
        
    except requests.exceptions.RequestException as e:
        return {"error": f"Azure API request failed: {str(e)}"}
    except Exception as e:
        return {"error": f"Unexpected error: {str(e)}"}

print("✅ Multi-image analysis function ready")

✅ Multi-image analysis function ready


In [14]:
print("✅ Using prompt from charles_bridge.md file for multi-image analysis")

✅ Using prompt from charles_bridge.md file for multi-image analysis


In [16]:
# Test analysis against annotated dataset
print(f"📋 Found {len(annotations)} annotated image pairs")

for i, annotation in enumerate(annotations):
    image_names = annotation['image_names']
    expected_crowdedness = annotation['crowdedness_score']
    
    print(f"\n🧪 Testing pair {i+1}/{len(annotations)}:")
    print(f"  📸 Images: {image_names}")
    print(f"  🎯 Expected crowdedness: {expected_crowdedness}/10")
    
    # Build full paths to images
    image_paths = [str(image_dir / img_name) for img_name in image_names]
    
    # Check if images exist
    missing_images = [path for path in image_paths if not Path(path).exists()]
    if missing_images:
        print(f"  ❌ Missing images: {[Path(p).name for p in missing_images]}")
        continue
    
    # Analyze the pair
    result_pair = analyze_bridge_images_pair(image_paths, charles_bridge_prompt_content)
    
    if "error" in result_pair:
        print(f"  ❌ Analysis error: {result_pair['error']}")
    else:
        print("  ✅ Analysis completed!")
        
        if "analysis" in result_pair and isinstance(result_pair["analysis"], dict):
            analysis = result_pair["analysis"]
            predicted_crowdedness = analysis.get('overcrowdedness_level', None)
            
            if predicted_crowdedness is not None:
                difference = abs(predicted_crowdedness - expected_crowdedness)
                print(f"  🔍 Predicted crowdedness: {predicted_crowdedness}/10")
                print(f"  📊 Difference: {difference} (Expected: {expected_crowdedness})")
                print(f"  🌤️ Weather: {analysis.get('weather', 'N/A')}")
                print(f"  🕐 Time: {analysis.get('time_of_day', 'N/A')}")
                
                # Show accuracy
                if difference <= 1:
                    print("  🎯 GOOD: Within 1 point of expected")
                elif difference <= 2:
                    print("  ⚠️ OK: Within 2 points of expected")
                else:
                    print("  ❌ POOR: More than 2 points difference")
            else:
                print("  ⚠️ Could not extract crowdedness level from response")
                print("  📝 Raw response:", result_pair.get("raw_response", "")[:200])
        else:
            print("  ⚠️ Could not parse analysis as JSON")
            print("  📝 Raw response:", result_pair.get("raw_response", "")[:200])

print(f"\n✅ Testing completed for {len(annotations)} annotated pairs")

📋 Found 2 annotated image pairs

🧪 Testing pair 1/2:
  📸 Images: ['camera_101200_20251129_120603.jpg', 'camera_101201_20251129_120604.jpg']
  🎯 Expected crowdedness: 9/10
🔄 Analyzing 2 images: camera_101200_20251129_120603.jpg, camera_101201_20251129_120604.jpg


  ✅ Analysis completed!
  🔍 Predicted crowdedness: 7/10
  📊 Difference: 2 (Expected: 9)
  🌤️ Weather: overcast
  🕐 Time: afternoon
  ⚠️ OK: Within 2 points of expected

🧪 Testing pair 2/2:
  📸 Images: ['camera_101200_20251125_095232.jpg', 'camera_101201_20251125_095233.jpg']
  🎯 Expected crowdedness: 2/10
🔄 Analyzing 2 images: camera_101200_20251125_095232.jpg, camera_101201_20251125_095233.jpg
  ✅ Analysis completed!
  🔍 Predicted crowdedness: 3/10
  📊 Difference: 1 (Expected: 2)
  🌤️ Weather: overcast and foggy
  🕐 Time: morning
  🎯 GOOD: Within 1 point of expected

✅ Testing completed for 2 annotated pairs
  ✅ Analysis completed!
  🔍 Predicted crowdedness: 3/10
  📊 Difference: 1 (Expected: 2)
  🌤️ Weather: overcast and foggy
  🕐 Time: morning
  🎯 GOOD: Within 1 point of expected

✅ Testing completed for 2 annotated pairs
